In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types
from pyspark.sql import functions as F

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .config("spark.driver.extraJavaOptions", "-Djava.library.path=C:/tools/hadoop/bin") \
    .getOrCreate()

In [21]:
df_green = spark.read.parquet('../data/pq/green/2020/*')

In [22]:
df_yellow = spark.read.parquet('../data/pq/yellow/2020/*')

In [23]:
# df_green.show()
# df_green.printSchema()
# df_green.columns

In [24]:
# df_yellow.show()
# df_yellow.printSchema()
# df_yellow.columns

In [25]:
df_green = df_green \
    .withColumnRenamed('lpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('lpep_dropoff_datetime', 'dropoff_datetime')

In [26]:
df_yellow = df_yellow \
    .withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime')

In [27]:
# set(df_green.columns) & set(df_yellow.columns)

In [28]:
common_columns = []

yellow_columns = set(df_yellow.columns)

for col in df_green.columns:
    if col in yellow_columns:
        common_columns.append(col)

In [29]:
# common_columns

In [30]:
df_green_sel = df_green \
    .select(common_columns) \
    .withColumn('service_type', F.lit('green'))

In [31]:
df_yellow_sel = df_yellow \
    .select(common_columns) \
    .withColumn('service_type', F.lit('yellow'))

In [32]:
df_trips_data = df_green_sel.unionAll(df_yellow_sel)

In [33]:
# df_trips_data.groupBy('service_type').count().show()

In [34]:
df_trips_data.createOrReplaceTempView('trips_data')

In [35]:
'''
spark.sql("""
select
    service_type,
    count(1)
from trips_data
group by
    service_type
""").show()
'''

'\nspark.sql("""\nselect\n    service_type,\n    count(1)\nfrom trips_data\ngroup by\n    service_type\n""").show()\n'

In [36]:
df_result = spark.sql("""
SELECT
    -- Revenue grouping
    PULocationID AS revenue_zone,
    date_trunc('month', pickup_datetime) AS revenue_month,
    service_type,

    -- Revenue calculation
    SUM(fare_amount) AS revenue_monthly_fare,
    SUM(extra) AS revenue_monthly_extra,
    SUM(mta_tax) AS revenue_monthly_mta_tax,
    SUM(tip_amount) AS revenue_monthly_tip_amount,
    SUM(tolls_amount) AS revenue_monthly_tolls_amount,
    SUM(improvement_surcharge) AS revenue_monthly_improvement_surcharge,
    SUM(total_amount) AS revenue_monthly_total_amount,
    SUM(congestion_surcharge) AS revenue_monthly_congestion_surcharge,

    -- Additional calculations
    AVG(passenger_count) AS avg_monthly_passenger_count,
    AVG(trip_distance) AS avg_monthly_trip_distance
FROM
    trips_data
GROUP BY
    1, 2, 3
""")

In [37]:
# df_result.show()

In [38]:
df_result.write.parquet('../data/report/revenue_monthly/', mode='overwrite')